# HW 2: Wrangling

**Q1.** This question provides some practice cleaning variables which have common problems.
1. Numeric variable: For `./data/airbnb_hw.csv`, clean the `Price` variable as well as you can, and explain the choices you make. How many missing values do you end up with? (Hint: What happens to the formatting when a price goes over 999 dollars, say from 675 to 1,112?)
2. Categorical variable: For the Minnesota police use of for data, `./data/mn_police_use_of_force.csv`, clean the `subject_injury` variable, handling the NA's; this gives a value `Yes` when a person was injured by police, and `No` when no injury occurred. What proportion of the values are missing? Is this a concern? Cross-tabulate your cleaned `subject_injury` variable with the `force_type` variable. Are there any patterns regarding when the data are missing?
3. Dummy variable: For the pretrial data covered in the lecture, clean the `WhetherDefendantWasReleasedPretrial` variable as well as you can, and, in particular, replace missing values with `np.nan`.
4. Missing values, not at random: For the pretrial data covered in the lecture, clean the `ImposedSentenceAllChargeInContactEvent` variable as well as you can, and explain the choices you make. (Hint: Look at the `SentenceTypeAllChargesAtConvictionInContactEvent` variable.)

In [ ]:
import pandas as pd
import numpy as np

# 1.Numeric variable: Price
df_airbnb = pd.read_csv('./data/airbnb_hw.csv')
# Remove '$' and ',' then convert to numeric
df_airbnb['Price'] = df_airbnb['Price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df_airbnb['Price'] = pd.to_numeric(df_airbnb['Price'], errors='coerce')
print(f"Missing values in Price: {df_airbnb['Price'].isna().sum()}")

# 2.Categorical variable: subject_injury
df_police = pd.read_csv('./data/mn_police_use_of_force.csv')
missing_prop = df_police['subject_injury'].isna().mean()
print(f"Proportion missing subject_injury: {missing_prop:.2%}")
# Cross-tabulate
print(pd.crosstab(df_police['subject_injury'].fillna('Missing'), df_police['force_type']))

# 3.Dummy variable: WhetherDefendantWasReleasedPretrial
df_justice = pd.read_parquet('./data/justice_data.parquet')
df_justice['WhetherDefendantWasReleasedPretrial'] = df_justice['WhetherDefendantWasReleasedPretrial'].replace('', np.nan)

# 4.Missing values: ImposedSentenceAllChargeInContactEvent
# Inspect relationship with SentenceTypeAllChargesAtConvictionInContactEvent
# If no conviction, likely no sentence imposed.
df_justice['ImposedSentenceAllChargeInContactEvent'] = pd.to_numeric(df_justice['ImposedSentenceAllChargeInContactEvent'], errors='coerce')


**Q2.** Go to https://sharkattackfile.net/ and download their dataset on shark attacks.

1. Open the shark attack file using Pandas. It is probably not a csv file, so `read_csv` won't work.
2. Drop any columns that do not contain data.
3. Clean the year variable. Describe the range of values you see. Filter the rows to focus on attacks since 1940. Are attacks increasing, decreasing, or remaining constant over time?
4. Clean the Age variable and make a histogram of the ages of the victims.
5. What proportion of victims are male?
6. Clean the `Type` variable so it only takes three values: Provoked and Unprovoked and Unknown. What proportion of attacks are unprovoked?
7. Clean the `Fatal Y/N` variable so it only takes three values: Y, N, and Unknown.
8. Are sharks more likely to launch unprovoked attacks on men or women? Is the attack more or less likely to be fatal when the attack is provoked or unprovoked? Is it more or less likely to be fatal when the victim is male or female? How do you feel about sharks?
9. What proportion of attacks appear to be by white sharks? (Hint: `str.split()` makes a vector of text values into a list of lists, split by spaces.)

In [ ]:
#Q2 Sharks Analysis
#1. Open shark attack file
#Using latin1 or cp1252 as common for this dataset
try:
    df_sharks = pd.read_csv('../data/sharks.csv', encoding='ISO-8859-1')
except:
    df_sharks = pd.read_csv('../data/sharks.csv', encoding='cp1252')

#2.Drop columns with no data
df_sharks = df_sharks.dropna(how='all', axis=1)

#3.Clean Year
df_sharks['Year'] = pd.to_numeric(df_sharks['Year'], errors='coerce')
df_sharks = df_sharks.dropna(subset=['Year'])
df_sharks = df_sharks[df_sharks['Year'] >= 1940]
print("Attack trend since 1940:")
counts = df_sharks['Year'].value_counts().sort_index()
print(counts.tail())

#4.Clean Age
df_sharks['Age'] = pd.to_numeric(df_sharks['Age'], errors='coerce')
print("Age description:\n", df_sharks['Age'].describe())

#5.Proportion Male
sex_col = 'Sex'
if 'Unnamed: 9' in df_sharks.columns and 'Sex' not in df_sharks.columns:
    df_sharks = df_sharks.rename(columns={'Unnamed: 9': 'Sex'})

if 'Sex' in df_sharks.columns:
    df_sharks['Sex'] = df_sharks['Sex'].astype(str).str.strip().str.upper()
    df_sharks['Sex'] = df_sharks['Sex'].replace({'M ': 'M', 'F ': 'F'})
    prop_male = (df_sharks['Sex'] == 'M').mean()
    print(f"Proportion Male: {prop_male:.2%}")

#6.Clean Type
valid_types = ['Provoked', 'Unprovoked']
df_sharks['Type'] = df_sharks['Type'].astype(str).str.strip()
df_sharks.loc[~df_sharks['Type'].isin(valid_types), 'Type'] = 'Unknown'
prop_unprovoked = (df_sharks['Type'] == 'Unprovoked').mean()
print(f"Proportion Unprovoked: {prop_unprovoked:.2%}")

#7.Clean Fatal Y/N
fatal_col_candidates = [c for c in df_sharks.columns if 'Fatal' in c]
if fatal_col_candidates:
    fatal_col = fatal_col_candidates[0]
    df_sharks[fatal_col] = df_sharks[fatal_col].astype(str).str.strip().str.upper()
    valid_fatal = ['Y', 'N']
    df_sharks.loc[~df_sharks[fatal_col].isin(valid_fatal), fatal_col] = 'Unknown'
else:
    fatal_col = None

#8.Comparisons
if 'Sex' in df_sharks.columns:
    print("\nSex vs Type (Unprovoked?):\n", pd.crosstab(df_sharks['Sex'], df_sharks['Type']))

if fatal_col:
    print("\nFatal vs Type:\n", pd.crosstab(df_sharks[fatal_col], df_sharks['Type']))
    if 'Sex' in df_sharks.columns:
        print("\nFatal vs Sex:\n", pd.crosstab(df_sharks[fatal_col], df_sharks['Sex']))

#9.White sharks
species_col_candidates = [c for c in df_sharks.columns if 'Species' in c]
if species_col_candidates:
    species_col = species_col_candidates[0]
    df_sharks[species_col] = df_sharks[species_col].astype(str).fillna('')
    white_shark_mask = df_sharks[species_col].str.contains('white', case=False, regex=True)
    prop_white = white_shark_mask.mean()
    print(f"Proportion White Shark attacks: {prop_white:.2%}")


**Q3.** Open the "tidy_data.pdf" document in the repo, which is a paper called Tidy Data by Hadley Wickham.

  1. Read the abstract. What is this paper about?
  
  **Answer**: The paper is about tidy data, which is a standard way of organizing data values within a dataset to make analysis easier and efficient and defines tidy data structure and tools

  2. Read the introduction. What is the "tidy data standard" intended to accomplish?
  
  **Answer**:The goal is to facilitate the data tidy process by providing a standard strcture that maps dataset to its physical layour which then we can use tidy tools that automate analysis and modeling.

  3. Read the intro to section 2. What does this sentence mean: "Like families, tidy datasets are all alike but every messy dataset is messy in its own way." What does this sentence mean: "For a given dataset, it’s usually easy to figure out what are observations and what are variables, but it is surprisingly difficult to precisely define variables and observations in general.
  
  **Answer**: It means that you can use the same tools to analyze different datasets.. Messy datasets are unique so each one requires its own cleaning script. "For a given dataset" means that when you look at a specific data it's easy to figure out what are observations and what are variables, but it is surprisingly difficult to precisely define variables and observations in general.

  4. Read Section 2.2. How does Wickham define values, variables, and observations?
  
  **Answer**: Values are the data points, variables are the columns, and observations are the rows.
  Variables are atrributes that measure the same underlying quality across units
  Observations are units being measured. 

  5. How is "Tidy Data" defined in section 2.3?
  
  **Answer**: Tidy data is defined by: Each variable forms a clumn, each observationf orms a row, each type of observational unit forms a table

  6. Read the intro to Section 3 and Section 3.1. What are the 5 most common problems with messy datasets? Why are the data in Table 4 messy? What is "melting" a dataset?
  
  **Answer**: The 5 most common problems with messy datasets are: 
  Column headers are values, not variable names
  Multiple variables are stored in one column
  Variables are stored in both rows and columns
  Multiple types of observational units are stored in the ssame table
  A single observational unit is stored in multiple tables
  Table 4 is messy, column headers are values rather than variable names.
  Melting a dataset means converting a dataset from wide format to long format.


  
  7. Why, specifically, is table 11 messy but table 12 tidy and "molten"?
  
  **Answer**: Table 11 is messy, column headers are values rather than variable names.
  Table 12 is tidy and "molten" because it has each variable forms a column, each observation forms a row, each type of observational unit forms a table.

  8. Read Section 6. What is the "chicken-and-egg" problem with focusing on tidy data? What does Wickham hope happens in the future with further work on the subject of data wrangling?
  
  **Answer**: The "chicken-and-egg" problem with focusing on tidy data is that it is not clear which approach is better. Wickham hopes that in the future, the subject of data wrangling will be more focused on tidy data.